# ⚖️ Ajuste de Umbral — Punto de Operación orientado a Sensibilidad

Carga el modelo binario de zona ya entrenado (`mejor_zona_binario.pt`) y, **sin reentrenar**, explora distintos umbrales de decisión para subir la **sensibilidad** (detección de zona de riesgo), aprovechando que la especificidad actual (0,97) tiene mucho margen.

**Lógica clínica:** en una red de seguridad, omitir un caso de riesgo (falso negativo) es más grave que una falsa alarma (falso positivo). Por eso conviene bajar el umbral para detectar más riesgo, sacrificando algo de especificidad.

**Honestidad:** el punto de operación recomendado se elige en **validación**, no en test. La tabla sobre test se muestra solo para transparencia del intercambio.

## 1 · Librerías y carga del modelo

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, pandas as pd, sys
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score
sys.path.append('..')
from src.preprocessing import MAX_LENGTH

SEED=42
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
MODELO='PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'; MODELO_ALT='BSC-LT/roberta-base-biomedical-clinical-es'
RUTA='../results/mejor_zona_binario.pt'
print("Dispositivo:", DEVICE)

## 2 · Datos (mismo split, reagrupado a zona)

In [ ]:
df = pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
X = df['auditor_input'].values; y7 = df['BI-RADS'].values
X_tv, X_test, y7_tv, y7_test = train_test_split(X, y7, test_size=0.15, random_state=SEED, stratify=y7)
X_train, X_val, y7_train, y7_val = train_test_split(X_tv, y7_tv, test_size=0.176, random_state=SEED, stratify=y7_tv)
z = lambda v: (np.asarray(v) >= 4).astype(int)
y_val, y_test = z(y7_val), z(y7_test)
print(f"Val: {len(X_val)} (riesgo: {y_val.sum()}) | Test: {len(X_test)} (riesgo: {y_test.sum()})")

## 3 · Reconstruir el modelo y cargar pesos

In [ ]:
try: tokenizer=AutoTokenizer.from_pretrained(MODELO)
except Exception: MODELO=MODELO_ALT; tokenizer=AutoTokenizer.from_pretrained(MODELO)

class ZonaDataset(Dataset):
    def __init__(self,t,l,tok,ml=MAX_LENGTH): self.t=list(t); self.l=list(l); self.tok=tok; self.ml=ml
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(str(self.t[i]),truncation=True,padding='max_length',max_length=self.ml,
                   return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}

class ZonaRoBERTa(nn.Module):
    def __init__(self, modelo, dropout=0.3):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(modelo)
        self.dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(self.encoder.config.hidden_size, 2)
    def forward(self, input_ids, attention_mask):
        out=self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(out.last_hidden_state[:,0,:]))

val_loader =DataLoader(ZonaDataset(X_val,y_val,tokenizer),batch_size=16)
test_loader=DataLoader(ZonaDataset(X_test,y_test,tokenizer),batch_size=16)
model=ZonaRoBERTa(MODELO).to(DEVICE)
model.load_state_dict(torch.load(RUTA,map_location=DEVICE)); model.eval()
print("Modelo cargado desde", RUTA)

## 4 · Calcular probabilidades de riesgo

In [ ]:
def probs_labels(loader):
    ps,ls=[],[]
    with torch.no_grad():
        for b in loader:
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
            ps.extend(F.softmax(model(ids,mask),dim=1)[:,1].cpu().numpy()); ls.extend(b['labels'].numpy())
    return np.array(ps), np.array(ls)

p_val,l_val = probs_labels(val_loader)
p_test,l_test = probs_labels(test_loader)
print(f"AUC val: {roc_auc_score(l_val,p_val):.4f} | AUC test: {roc_auc_score(l_test,p_test):.4f}")

## 5 · Tabla de intercambio (barrido de umbrales en TEST)

Solo para transparencia: muestra cómo cambian sensibilidad y especificidad según el umbral.

In [ ]:
def met(y,p,thr):
    yp=(p>=thr).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    sens=tp/(tp+fn) if (tp+fn) else 0; espec=tn/(tn+fp) if (tn+fp) else 0
    vpp=tp/(tp+fp) if (tp+fp) else 0
    return sens,espec,vpp,tp,fn,fp

print(f"{'Umbral':>7} | {'Sensib.':>8} | {'Especif.':>8} | {'VPP':>6} | {'Riesgo det.':>11} | {'F.alarmas':>9}")
print("-"*65)
for thr in np.arange(0.10, 0.91, 0.05):
    s,e,v,tp,fn,fp = met(l_test,p_test,thr)
    print(f"{thr:>7.2f} | {s:>8.3f} | {e:>8.3f} | {v:>6.3f} | {tp:>4}/{tp+fn:<6} | {fp:>9}")

## 6 · Punto de operación orientado a sensibilidad (elegido en VALIDACIÓN)

Se busca en validación el umbral más alto que logre una sensibilidad objetivo, y se aplica al test.

In [ ]:
OBJETIVO_SENS = 0.80   # objetivo de sensibilidad en validación (ajustable)

# Buscar en VAL el umbral más alto que alcance la sensibilidad objetivo
thr_elegido = 0.5
for thr in np.arange(0.05, 0.95, 0.01):
    s_val,_,_,_,_,_ = met(l_val,p_val,thr)
    if s_val >= OBJETIVO_SENS:
        thr_elegido = thr   # seguimos subiendo mientras se cumpla -> el más alto que cumple
print(f"Umbral elegido en validación (sens>={OBJETIVO_SENS}): {thr_elegido:.2f}\n")

for nombre,thr in [("Por defecto (0.50)",0.5),(f"Orientado a sensibilidad ({thr_elegido:.2f})",thr_elegido)]:
    s,e,v,tp,fn,fp = met(l_test,p_test,thr)
    print(f"{nombre} — en TEST:")
    print(f"   Sensibilidad: {s:.3f} | Especificidad: {e:.3f} | VPP: {v:.3f}")
    print(f"   Riesgo detectado: {tp}/{tp+fn} | No detectados: {fn} | Falsas alarmas: {fp}\n")

## 7 · Cómo reportarlo (honestidad)

- Reportar el **AUC (0,93)** como métrica principal (independiente del umbral).
- Presentar el **punto de operación orientado a sensibilidad** como una **decisión clínica explícita**: "para una red de seguridad se privilegia la sensibilidad, aceptando más falsas alarmas".
- Dejar claro que el umbral se eligió en **validación**, no en test.
- Mantener la limitación honesta: con ~11 casos de riesgo en test, las métricas tienen incertidumbre amplia; es una **prueba de concepto**.
- Mostrar la tabla de intercambio da transparencia: el lector ve que no se "eligió a dedo" el punto que más conviene.